### Library and model imports (execute just once)

In [ ]:
from utils import list_images, read_image, remove_border_cells, extract_img_metadata
from cellpose import models, core
from tifffile import imread
from pathlib import Path
import numpy as np
import napari
import matplotlib.pyplot as plt

# Ignore OME-TIFF warnings during imread
import logging
logging.getLogger("tifffile").setLevel(logging.ERROR)

#Check if notebook has GPU access
if core.use_gpu()==False:
  raise ImportError("No GPU access, change your runtime")

#Load pre-trained Cellpose-SAM
model = models.CellposeModel(gpu=True)

### Define the path containing your images

In [ ]:
# Copy the path to the folder containing your images in between the quotation marks
data_folder = r"X:\Shirin\RCDAnalysis_Alberto\SK0047_Exp01-02"

# If the path is correct you should see a list of the first 10 images in your folder down below
images = list_images(data_folder, format="nd2")
images[:10]

### Define a single image file for exploration
Just edit the number in between brackets [ ] under <code>img_filepath = images[0]</code> to select the file

In [ ]:

# Extract experiment_id from data folder Path object
experiment_id = Path(data_folder).name

# Open any of the  images to analyze by changing the index number in between brackets []
# index number refers to the position of the file inside the images folder
# i.e. to open the first image [0], second [1] - in Python one starts counting from zero
img_filepath = images[0]
img = imread(img_filepath)

### Predict nuclei and cell labels using base CellposeSAM (4.0)

In [ ]:
# Extract image metadata from filename
descriptor_dict = extract_img_metadata (img_filepath, verbose = True)

# Predict nuclei labels using CellposeSAM 
nuclei_labels, flows, styles = model.eval(img[1], niter=1000) # need to check the arguments

# Predict cell labels using CellposeSAM using nuclei (1) and brightfield (2) images as input 
cell_labels, flows, styles = model.eval((img[[1,2]]), niter=1000) # need to check the arguments

# Visualize results in Napari
viewer = napari.Viewer(ndisplay=2)
viewer.add_image(img)

# Label-to-label remapping: each nucleus inherits the cell label value it lies in
nuclei_remapped = remap_labels(nuclei_labels, cell_labels)

# Remove cell entities touching the image border
cell_labels, nuclei_remapped = remove_border_cells(cell_labels, nuclei_remapped) 
viewer.add_labels(cell_labels, opacity=0.5)
viewer.add_labels(nuclei_remapped, name="nuclei_labels", opacity=0.8)

props_df = extract_features(img, nuclei_remapped, cell_labels, descriptor_dict, scikit_props=["label", "intensity_mean", "area"])
props_df


### Sanity check of generated label areas and threshold selection

In [ ]:

props_df["cell_to_nuclei_ratio"].dropna().plot.hist(bins=100)
plt.show()

# Check individual area values to set a logical threshold
print(props_df[props_df["label"] == 300])

# Filter out labels with a cell to nuclei ratio smaller than 2
# These are cells with either incorrect nuclei segmentation or very clustered together
props_df = props_df[props_df["cell_to_nuclei_ratio"] >= 2]


props_df["cell_to_nuclei_ratio"].dropna().plot.hist(bins=100)
plt.show()


### Visual inspection of nuclei and cell labels after filtering

In [ ]:
# Collect valid labels
valid_labels = props_df["label"].values

# Mask nuclei_labels: keep only valid labels (according to cell to nuclei ratio)
nuclei_labels_filtered = nuclei_remapped.copy()
nuclei_labels_filtered[~np.isin(nuclei_labels_filtered, valid_labels)] = 0

# Mask cell_labels: keep only valid labels (according to cell to nuclei ratio)
cell_labels_filtered = cell_labels.copy()
cell_labels_filtered[~np.isin(cell_labels_filtered, valid_labels)] = 0

viewer.add_labels(nuclei_labels_filtered)
viewer.add_labels(cell_labels_filtered)